# EDA dan Visualisasi Interaktif

Melakukan Exploratory Data Analysis (EDA) dan menghasilkan minimal 6 visualisasi interaktif dan 5 insight berdasarkan dataset Flood Prediction South Sulawesi.

In [37]:
import os
import polars as pl
import plotly.express as px

CLEAN_DATA_PATH = '../data/modis_flood_features_clean.parquet'
if not os.path.exists(CLEAN_DATA_PATH):
    CLEAN_DATA_PATH = 'TUGAS1/data/modis_flood_features_clean.parquet'

# Load data bersih lazily dan konversi ke Pandas untuk plotting
df = pl.scan_parquet(CLEAN_DATA_PATH).collect()
df_pd = df.to_pandas()

## 1. Analisis Spasial: Peta Lokasi Kejadian Banjir

In [38]:
# Sampling agar peta tidak terlalu berat untuk dirender
df_flood = df_pd[df_pd['target'] == 1].dropna(subset=['lat', 'lon'])
df_map_sample = df_flood.sample(n=1000, random_state=42) if len(df_flood) > 1000 else df_flood

fig1 = px.scatter_map(df_map_sample, lat='lat', lon='lon', color='precip_3d',
                         size_max=10, hover_name='target',
                         map_style='carto-positron', zoom=6, title='Peta Lokasi Kejadian Banjir (Target=1)')
fig1.show()

**Insight 1:** Visualisasi peta menunjukkan lokasi sebaran area yang mengalami banjir. Titik yang lebih gelap menunjukkan curah hujan (precip_3d) yang lebih tinggi di wilayah tersebut.

## 2. Hubungan Curah Hujan dan Elevasi Tanah

In [39]:
df_sample = df_pd.sample(n=2000, random_state=42) if len(df_pd) > 2000 else df_pd
df_sample['target_str'] = df_sample['target'].astype(str)
fig2 = px.scatter(df_sample, x='elevation', y='precip_3d', color='target_str', 
                  title='Curah Hujan (3 Hari) vs Elevasi Tanah', labels={'target_str': 'Flooded'})
fig2.show()

**Insight 2:** Terdapat pola bahwa kejadian banjir (Flooded=1) cenderung terjadi pada area dengan elevasi rendah, meskipun ada curah hujan yang cukup tinggi.

## 3. Distribusi Curah Hujan Harian

In [40]:
fig3 = px.histogram(df_sample, x='precip_1d', color='target_str', nbins=50, 
                    title='Distribusi Curah Hujan 1 Hari (precip_1d)', barmode='overlay')
fig3.show()

**Insight 3:** Distribusi curah hujan sangat condong ke kanan (right-skewed). Hari-hari dengan curah hujan sangat ekstrem memiliki probabilitas banjir yang jauh lebih tinggi.

## 4. Rata-rata Elevasi Berdasarkan Status Banjir

In [41]:
elev_by_target = df_pd.groupby('target')['elevation'].mean().reset_index()
elev_by_target['target'] = elev_by_target['target'].astype(str)
fig4 = px.bar(elev_by_target, x='target', y='elevation', title='Rata-rata Elevasi Tanah Berdasarkan Status Banjir')
fig4.show()

**Insight 4:** Rata-rata ketinggian tanah pada wilayah yang mengalami banjir (Target=1) secara signifikan lebih rendah dibandingkan wilayah yang tidak banjir (Target=0).

## 5. Proporsi Kejadian Banjir Secara Keseluruhan

In [42]:
target_counts = df_pd['target'].value_counts().reset_index()
target_counts.columns = ['target_status', 'count']
target_counts['target_status'] = target_counts['target_status'].apply(lambda x: 'Banjir' if x == 1 else 'Tidak Banjir')
fig5 = px.pie(target_counts, names='target_status', values='count', title='Proporsi Kejadian Banjir')
fig5.show()

**Insight 5:** Kejadian banjir merupakan anomali atau data minoritas, sehingga persentase wilayah/hari yang mengalami banjir jauh lebih kecil dibandingkan yang tidak banjir (imbalanced data).

## 6. Distribusi Indeks Topografi (TWI)

In [43]:
fig6 = px.box(df_sample, x='target_str', y='TWI', title='Distribusi Topographic Wetness Index (TWI)', 
              labels={'target_str': 'Status Banjir (1=Banjir)'})
fig6.show()

**Insight Tambahan:** Topographic Wetness Index (TWI) yang merepresentasikan kecenderungan akumulasi air secara topografis menunjukkan nilai median yang lebih tinggi pada area yang dilanda banjir.